# RQ1: Bias trigger detection analysis

Requires the outputs of `bash_scripts/rq1_detection.sh` (`outputs/bias_detection/`). Overall per-model scores are in each model's `classification_report_<model>.csv`.

Tables:
1. F1 per bias type
2. Macro-F1 on single-bias vs. pairwise-bias sentences
3. Macro-F1 per pairwise combination
4. Precision / recall / F1 of the `none` class

In [ ]:
import os
from itertools import combinations
from pathlib import Path

import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer

DETECTION_DIR = "../outputs/bias_detection"
ANALYSIS_DIR = "../outputs/analysis"
os.makedirs(ANALYSIS_DIR, exist_ok=True)

MODELS = ["gpt-5.4-mini", "claude-haiku-4.5", "gemini-3-flash", "llama-3.1-8b", "llama-3.1-70b", "llama-3.3-70b",
          "qwen3-8b", "qwen3.5-9b", "qwen3-32b", "gemma-4-12b", "gemma-4-31b"]

BIAS_TYPES = ['anchoring', 'authority', 'framing', 'herding', 'recency', 'overconfidence', 'none']
BIAS_ONLY = ['anchoring', 'authority', 'framing', 'herding', 'recency', 'overconfidence']

In [ ]:
def parse_labels(text: str) -> set:
    if pd.isna(text) or str(text).strip() == "" or str(text).strip().lower() == "none":
        return {"none"}
    return {b.strip().lower() for b in str(text).split(',') if b.strip()}

def parse_labels_no_none(text: str) -> set:
    if pd.isna(text) or str(text).strip() == "" or str(text).strip().lower() == "none":
        return set()
    return {b.strip().lower() for b in str(text).split(',') if b.strip() and b.strip().lower() != "none"}

def get_pairwise_combinations():
    non_framing = ['anchoring', 'authority', 'herding', 'recency', 'overconfidence']
    pairs = list(combinations(non_framing, 2))
    for bias in non_framing:
        pairs.append(('framing', bias))
    return pairs

def load_model_predictions(model_folder: str):
    predictions_file = Path(DETECTION_DIR) / model_folder / f"dataset_predictions_{model_folder}.csv"
    if predictions_file.exists():
        return pd.read_csv(predictions_file)
    return None

## Table 1: F1 per bias type

In [ ]:
per_bias_results = {bias: {} for bias in BIAS_TYPES}

for model in MODELS:
    df = load_model_predictions(model)
    if df is None:
        for bias in BIAS_TYPES:
            per_bias_results[bias][model] = "TBD"
        continue
    
    mlb = MultiLabelBinarizer(classes=BIAS_TYPES)
    y_true = df["biases"].apply(parse_labels).tolist()
    y_pred = df["predicted_biases"].apply(parse_labels).tolist()
    
    y_true_bin = mlb.fit_transform(y_true)
    y_pred_bin = mlb.transform(y_pred)
    
    f1_per_class = f1_score(y_true_bin, y_pred_bin, average=None, zero_division=0)
    for i, bias in enumerate(mlb.classes_):
        per_bias_results[bias][model] = round(f1_per_class[i], 3)

per_bias_df = pd.DataFrame(per_bias_results).T
per_bias_df = per_bias_df[MODELS]
per_bias_df.index.name = "Bias Type"

per_bias_path = os.path.join(ANALYSIS_DIR, "per_bias_type_f1.csv")
per_bias_df.to_csv(per_bias_path)
print(f"Saved: {per_bias_path}")
per_bias_df

## Table 2: Single vs. pairwise macro-F1 (bias classes only, unbiased sentences excluded)

In [ ]:
single_pairwise_results = {}

mlb_no_none = MultiLabelBinarizer(classes=BIAS_ONLY)
mlb_no_none.fit([BIAS_ONLY])

for model in MODELS:
    df = load_model_predictions(model)
    if df is None:
        single_pairwise_results[model] = {"Single": "TBD", "Pairwise": "TBD"}
        continue
    
    # Filter out none rows
    df = df[df["biases"] != "none"].copy()
    
    df["true_labels"] = df["biases"].apply(parse_labels_no_none)
    df["pred_labels"] = df["predicted_biases"].apply(parse_labels_no_none)
    df["num_true_biases"] = df["true_labels"].apply(len)
    
    single_df = df[df["num_true_biases"] == 1]
    pairwise_df = df[df["num_true_biases"] == 2]
    
    if len(single_df) > 0:
        y_true_single = mlb_no_none.transform(single_df["true_labels"].tolist())
        y_pred_single = mlb_no_none.transform(single_df["pred_labels"].tolist())
        single_f1 = round(f1_score(y_true_single, y_pred_single, average="macro", zero_division=0), 3)
    else:
        single_f1 = "TBD"
    
    if len(pairwise_df) > 0:
        y_true_pair = mlb_no_none.transform(pairwise_df["true_labels"].tolist())
        y_pred_pair = mlb_no_none.transform(pairwise_df["pred_labels"].tolist())
        pairwise_f1 = round(f1_score(y_true_pair, y_pred_pair, average="macro", zero_division=0), 3)
    else:
        pairwise_f1 = "TBD"
    
    single_pairwise_results[model] = {"Single": single_f1, "Pairwise": pairwise_f1}

single_pairwise_df = pd.DataFrame(single_pairwise_results).T
single_pairwise_df = single_pairwise_df.reindex(MODELS)
single_pairwise_df.index.name = "Model"

single_pairwise_path = os.path.join(ANALYSIS_DIR, "single_vs_pairwise_f1.csv")
single_pairwise_df.to_csv(single_pairwise_path)
print(f"Saved: {single_pairwise_path}")
single_pairwise_df

## Table 3: Macro-F1 per pairwise combination (over the two biases in each pair)

In [ ]:
pairwise_combos = get_pairwise_combinations()
combo_labels = [f"{b1},{b2}" for b1, b2 in pairwise_combos]
pairwise_combo_results = {combo: {} for combo in combo_labels}

for model in MODELS:
    df = load_model_predictions(model)
    if df is None:
        for combo in combo_labels:
            pairwise_combo_results[combo][model] = "TBD"
        continue
    
    df["true_labels"] = df["biases"].apply(parse_labels_no_none)
    df["pred_labels"] = df["predicted_biases"].apply(parse_labels_no_none)
    df["num_true_biases"] = df["true_labels"].apply(len)
    
    pairwise_df = df[df["num_true_biases"] == 2].copy()
    pairwise_df["combination"] = pairwise_df["true_labels"].apply(
        lambda x: ",".join(sorted(x))
    )
    
    for combo in combo_labels:
        combo_parts = set(combo.split(","))
        combo_key = ",".join(sorted(combo_parts))
        combo_biases = list(combo_parts)  # The 2 biases in this combination
        
        combo_df = pairwise_df[pairwise_df["combination"] == combo_key]
        
        if len(combo_df) > 0:
            # Use only the 2 biases present in this combination
            mlb_combo = MultiLabelBinarizer(classes=combo_biases)
            mlb_combo.fit([combo_biases])
            
            y_true = mlb_combo.transform(combo_df["true_labels"].tolist())
            y_pred = mlb_combo.transform(combo_df["pred_labels"].tolist())
            combo_f1 = round(f1_score(y_true, y_pred, average="macro", zero_division=0), 3)
        else:
            combo_f1 = "TBD"
        
        pairwise_combo_results[combo][model] = combo_f1

pairwise_combo_df = pd.DataFrame(pairwise_combo_results).T
pairwise_combo_df = pairwise_combo_df[MODELS]
pairwise_combo_df.index.name = "Combination"

pairwise_combo_path = os.path.join(ANALYSIS_DIR, "pairwise_combination_f1.csv")
pairwise_combo_df.to_csv(pairwise_combo_path)
print(f"Saved: {pairwise_combo_path}")
pairwise_combo_df

## Table 4: `none` class (binary: unbiased vs. any bias)

In [ ]:
results = []

for model in MODELS:
    df = load_model_predictions(model)
    
    if df is None:
        results.append({"model": model, "P": "TBD", "R": "TBD", "F1": "TBD"})
        continue
    
    # Binary: 1 if none, 0 otherwise
    y_true = (df["biases"] == "none").astype(int)
    y_pred = (df["predicted_biases"] == "none").astype(int)
    
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    results.append({
        "model": model,
        "P": round(precision, 3),
        "R": round(recall, 3),
        "F1": round(f1, 3)
    })

none_df = pd.DataFrame(results).set_index("model")

none_path = os.path.join(ANALYSIS_DIR, "none_class_scores.csv")
none_df.to_csv(none_path)
print(f"Saved: {none_path}")
none_df